# labvault クイックスタート

このNotebookでは labvault の基本操作を一通り体験します。  
InMemoryBackend で動くため、外部サービスは不要です。

## 1. Lab の初期化

`Lab` はチームのデータベースへのエントリポイントです。  
引数なしで作ると InMemoryBackend が自動で使われます。

In [ ]:
from labvault import Lab

lab = Lab("konishi-lab", user="taro")
lab

## 2. レコードの作成

`lab.new()` で実験レコードを作成します。  
条件は `**kwargs` で直接渡せます。

In [ ]:
exp = lab.new(
    "SiO2薄膜スパッタ成膜",
    tags=["スパッタ", "SiO2"],
    temperature_C=500,
    pressure_Pa=0.5,
    power_W=200,
)

print(f"ID: {exp.id}")
print(f"Status: {exp.status}")
print(f"Tags: {exp.tags}")
print(f"Conditions: {exp.get_conditions()}")

## 3. メモの追加

`note()` でタイムスタンプ付きメモを残せます。  
同じテキストを連続で呼んでも重複しません (冪等性)。

In [ ]:
exp.note("成膜前にチャンバー 30分ベーキング済み")
exp.note("基板洗浄: アセトン → IPA → UV オゾン 10分")

for n in exp.notes:
    print(f"  [{n.created_at:%H:%M:%S}] {n.text}")

## 4. ファイルの保存

`add()` でファイルパスやバイトデータを保存できます。  
`save()` を使うと Python オブジェクト (dict, list, str) を自動判定して保存します。

In [ ]:
# dict を JSON として保存
exp.save("parameters", {
    "target": "SiO2",
    "substrate": "Si(100)",
    "gas": "Ar",
    "flow_rate_sccm": 20,
})

# バイトデータ (測定データのつもり) を保存
dummy_csv = "time_s,thickness_nm\n0,0\n60,12.5\n120,25.1\n180,37.8\n"
exp.add(dummy_csv.encode(), name="thickness.csv", content_type="text/csv")

print("保存したファイル:")
for ref in exp.list_data():
    print(f"  {ref.name} ({ref.content_type}, {ref.size_bytes} bytes)")

## 5. ファイルの取得

`get_data(name)` で保存済みファイルをバイナリで取り出せます。  
`list_data()` でファイル一覧をメタデータ付きで確認できます。

In [ ]:
# 保存した CSV を取得して表示
csv_bytes = exp.get_data("thickness.csv")
print(csv_bytes.decode())

# JSON も取得できる
import json
params = json.loads(exp.get_data("parameters.json"))
print(f"Target: {params['target']}")

## 6. 結果の記録

`results` は dict ライクなプロキシです。値を代入するたびに自動で永続化されます。

In [ ]:
exp.results["film_thickness_nm"] = 37.8
exp.results["uniformity_percent"] = 2.3
exp.results["deposition_rate_nm_per_min"] = 12.6

print("Results:")
for k, v in exp.results.items():
    print(f"  {k}: {v}")

## 7. ステータスの変更

実験が完了したらステータスを更新します。  
(`with` 文を使うと自動でSUCCESS/FAILEDが設定されます。次のセクションで紹介)

In [ ]:
exp.status = "success"
print(exp)

## 8. with 文による自動ステータス管理

`with lab.new(...)` を使うと、正常終了で SUCCESS、例外で FAILED が自動設定されます。

In [ ]:
# 正常終了 → SUCCESS
with lab.new("XRD測定", tags=["XRD"]) as exp2:
    exp2.conditions(two_theta_range="20-80", step_deg=0.02)
    exp2.note("Cu-Kα線使用")
    exp2.save("xrd_data", {"2theta": [20, 30, 40], "intensity": [100, 500, 200]})

print(f"exp2: {exp2.status}")  # → success

In [ ]:
# 例外発生 → FAILED
try:
    with lab.new("失敗する実験") as exp3:
        raise RuntimeError("装置エラー")
except RuntimeError:
    pass

print(f"exp3: {exp3.status}")  # → failed

## 9. レコードの取得と一覧

作成したレコードは ID で取得したり、一覧で確認できます。

In [ ]:
# ID で取得
same_exp = lab.get(exp.id)
print(f"取得: {same_exp.title} (ID: {same_exp.id})")

# 最新レコード一覧
print("\n最近のレコード:")
for r in lab.recent():
    print(f"  [{r.id}] {r.title} ({r.status})")

## まとめ

labvault の基本操作:

| 操作 | メソッド |
|------|----------|
| レコード作成 | `lab.new(title, **conditions)` |
| 条件設定 | `exp.conditions(**kwargs)` |
| メモ追加 | `exp.note(text)` |
| ファイル保存 | `exp.add(path)` / `exp.save(name, obj)` |
| ファイル取得 | `exp.get_data(name)` |
| ファイル一覧 | `exp.list_data()` |
| 結果記録 | `exp.results[key] = value` |
| タグ | `exp.tag("XRD")` |
| 取得 | `lab.get(id)` |
| 一覧 | `lab.recent()` / `lab.list()` |